# Feature Engineering for Fraud Detection

This notebook creates transaction-level features from the raw payment dataset so the model can learn the patterns that differentiate fraudulent activity from normal transfers and cash movements.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "data" / "raw").exists():
    project_root = project_root.parent

raw_path = project_root / "data" / "raw" / "PS_20174392719_1491204439457_log.csv"
print(f"Project root: {project_root}")
print(f"Data path: {raw_path}")

df = pd.read_csv(raw_path)
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(df.head())


Project root: C:\Users\MY PC\Naresh IT\AI Projects store_fraud_detection\AI Projects store_fraud_detection
Data path: C:\Users\MY PC\Naresh IT\AI Projects store_fraud_detection\AI Projects store_fraud_detection\data\raw\PS_20174392719_1491204439457_log.csv
Rows: 6362620
Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0

In [2]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Basic transaction behavior
    df["amount_log"] = np.log1p(df["amount"])
    df["hour_of_day"] = df["step"] % 24
    df["is_cash_in"] = (df["type"] == "CASH_IN").astype(int)
    df["is_cash_out"] = (df["type"] == "CASH_OUT").astype(int)
    df["is_transfer"] = (df["type"] == "TRANSFER").astype(int)
    df["is_payment"] = (df["type"] == "PAYMENT").astype(int)
    df["is_debit"] = (df["type"] == "DEBIT").astype(int)

    # Account balance deltas
    df["balance_delta_orig"] = df["newbalanceOrig"] - df["oldbalanceOrg"]
    df["balance_delta_dest"] = df["newbalanceDest"] - df["oldbalanceDest"]

    # Relative size of the transaction compared to account balances
    df["amount_ratio_orig"] = np.where(
        df["oldbalanceOrg"] == 0,
        0.0,
        df["amount"] / df["oldbalanceOrg"],
    )
    df["amount_ratio_dest"] = np.where(
        df["oldbalanceDest"] == 0,
        0.0,
        df["amount"] / df["oldbalanceDest"],
    )

    # Flag suspicious patterns often associated with fraud
    df["is_zero_balance_origin"] = (df["oldbalanceOrg"] == 0).astype(int)
    df["is_zero_balance_dest"] = (df["oldbalanceDest"] == 0).astype(int)
    df["is_large_amount"] = (df["amount"] > df["amount"].quantile(0.99)).astype(int)
    df["is_large_transfer"] = (df["amount"] > 100000).astype(int)

    return df

feature_df = engineer_features(df)
print(feature_df[[
    "type", "amount", "oldbalanceOrg", "newbalanceOrig", "balance_delta_orig",
    "oldbalanceDest", "newbalanceDest", "balance_delta_dest", "amount_ratio_orig",
    "isFraud"
]].head())

print(f"Feature columns: {len(feature_df.columns)}")


       type    amount  oldbalanceOrg  newbalanceOrig  balance_delta_orig  \
0   PAYMENT   9839.64       170136.0       160296.36            -9839.64   
1   PAYMENT   1864.28        21249.0        19384.72            -1864.28   
2  TRANSFER    181.00          181.0            0.00             -181.00   
3  CASH_OUT    181.00          181.0            0.00             -181.00   
4   PAYMENT  11668.14        41554.0        29885.86           -11668.14   

   oldbalanceDest  newbalanceDest  balance_delta_dest  amount_ratio_orig  \
0             0.0             0.0                 0.0           0.057834   
1             0.0             0.0                 0.0           0.087735   
2             0.0             0.0                 0.0           1.000000   
3         21182.0             0.0            -21182.0           1.000000   
4             0.0             0.0                 0.0           0.280795   

   isFraud  
0        0  
1        0  
2        1  
3        1  
4        0  
Feature 

In [3]:
feature_columns = [
    "amount", "amount_log", "hour_of_day",
    "balance_delta_orig", "balance_delta_dest",
    "amount_ratio_orig", "amount_ratio_dest",
    "is_cash_in", "is_cash_out", "is_transfer", "is_payment", "is_debit",
    "is_zero_balance_origin", "is_zero_balance_dest",
    "is_large_amount", "is_large_transfer"
]

X = feature_df[feature_columns]
y = feature_df["isFraud"]

print("Feature engineering summary:")
print(X.describe().round(4))
print(f"Training rows: {len(X)}")
print(f"Fraud target mean: {y.mean():.4%}")


Feature engineering summary:
             amount    amount_log   hour_of_day  balance_delta_orig  \
count  6.362620e+06  6.362620e+06  6.362620e+06        6.362620e+06   
mean   1.798619e+05  1.084090e+01  1.532150e+01        2.123056e+04   
std    6.038582e+05  1.814500e+00  4.321800e+00        1.466433e+05   
min    0.000000e+00  0.000000e+00  0.000000e+00       -1.000000e+07   
25%    1.338957e+04  9.502300e+00  1.200000e+01       -1.015043e+04   
50%    7.487194e+04  1.122350e+01  1.600000e+01        0.000000e+00   
75%    2.087215e+05  1.224880e+01  1.900000e+01        0.000000e+00   
max    9.244552e+07  1.834210e+01  2.300000e+01        1.915268e+06   

       balance_delta_dest  amount_ratio_orig  amount_ratio_dest    is_cash_in  \
count        6.362620e+06       6.362620e+06       6.362620e+06  6.362620e+06   
mean         1.242947e+05       8.171350e+01       5.923600e+00  2.199000e-01   
std          8.129391e+05       3.395705e+03       2.941265e+03  4.142000e-01   
min    

In [4]:
processed_path = project_root / "data" / "processed" / "fraud_features.csv"
processed_path.parent.mkdir(parents=True, exist_ok=True)
feature_df.to_csv(processed_path, index=False)
print(f"Saved engineered dataset to: {processed_path}")
print(feature_df.head())


Saved engineered dataset to: C:\Users\MY PC\Naresh IT\AI Projects store_fraud_detection\AI Projects store_fraud_detection\data\processed\fraud_features.csv
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  ...  is_payment  \
0  M1979787155             0.0             0.0        0  ...           1   
1  M2044282225             0.0             0.0        0  ...           1   
2   C553264065             0.0             0.0        1  ...           0   
3    C38997010         21182.0             0.0        1  ...           0   
4  M123070170